# 汇总前面已计算好的A股交易数据

In [2]:
## 导入NumPy、Pandas、Wind
import numpy as np
import pandas as pd
from scipy import stats
from WindPy import w
import datetime
import statsmodels.api as sm
import matplotlib.pyplot as plt

In [2]:
## 启动Wind终端的API
## 判断WindPy是否已经登录成功
w.start() 
w.isconnected()

Welcome to use Wind Quant API for Python (WindPy)!

COPYRIGHT (C) 2020 WIND INFORMATION CO., LTD. ALL RIGHTS RESERVED.
IN NO CIRCUMSTANCE SHALL WIND BE RESPONSIBLE FOR ANY DAMAGES OR LOSSES CAUSED BY USING WIND QUANT API FOR Python.


True

## 市场月收益数据

In [3]:
## 导入领先一期的市场收益数据
mktr = w.wsd("881001.WI", "pct_chg", "2007-01-01", "2019-01-31", "Period=M")
mktr = pd.DataFrame(mktr.Data, index = mktr.Codes, columns = mktr.Times)
mktr = mktr.T
mktr = mktr.reset_index()
mktr.columns = ['date', 'pct_chg']
mktr['pct_chg'] = mktr['pct_chg'] * 0.01

In [4]:
## 调整收益率单位，生成匹配指标
mktr = mktr.reset_index()
mktr = mktr[['pct_chg', 'index']]

In [5]:
## 读取一月期Shibor
shibor = w.wsd("SHIBOR1M.IR", "open", "2007-01-01", "2019-01-31")
shibor = pd.DataFrame(shibor.Data,index = shibor.Codes,columns = shibor.Times)
shibor = shibor.T
shibor = shibor.reset_index()
shibor.columns = ['date', 'shibor']
shibor['date'] = pd.to_datetime(shibor['date'])
shibor['year'] = shibor['date'].dt.year
shibor['month'] = shibor['date'].dt.month
shibor['shibor'] = shibor['shibor'] * 0.01

In [6]:
## 计算各月shibor均值
intrst = shibor.groupby([shibor['year'], shibor['month']])['shibor'].agg([('mn_SHIBOR', 'mean'), ])
intrst = intrst.reset_index()
intrst = intrst.reset_index()
intrst.columns = ['index', 'year', 'month', 'shibor']

In [7]:
## 计算月超额收益率
mkt_exr = pd.merge(mktr, intrst, on = 'index', how = 'left')
mkt_exr['r'] = mkt_exr['pct_chg'] - mkt_exr['shibor']
mkt_exr = mkt_exr[['index', 'r']]

In [8]:
mkt_exr['r'].to_csv("Rgrss/r0.csv", index = False)

In [10]:
## 删除首行，得到领先一期的收益率
mkt_exr = mkt_exr.drop("index", axis = 1)
mkt_exr = mkt_exr.drop(0, axis = 0)
mkt_exr = mkt_exr.reset_index(drop = True)
mkt_exr = mkt_exr.reset_index()

## 读入宏观经济周期指标

In [2]:
fmacro = pd.read_csv("Rgrss/fmacro.csv")
fmacro = fmacro.reset_index()

## 市场收益方差、偏度、峰度

In [14]:
## 读取市场收益方差、偏度、峰度
mkt_prdct = pd.read_csv("Weight and Ave/mkt_prdct.csv")
mkt_prdct = mkt_prdct.reset_index()
mkt_prdct = mkt_prdct[['index', 'year', 'var', 'math_skew', 'math_kurt']]
mkt_prdct.columns = ['index', 'year', 'mkt_var', 'mkt_skew', 'mkt_kurt']

## 等权重加权法

In [15]:
## 等权重加权法下的市场平均方差、偏度、峰度
mnv_ew = pd.read_csv('Weight and Ave/mnv_ew.csv')
mnv_ew = mnv_ew.reset_index()
mnv_ew = mnv_ew[['index', 'var', 'math_skew', 'math_kurt']]
mnv_ew.columns = ['index', 'ave_var', 'ave_skew', 'ave_kurt']

In [24]:
## 等权重加权法下的市场非流动性
ILLIQE_ew = pd.read_csv("Rgrss/ILLIQE_ew.csv")
ILLIQE_ew = ILLIQE_ew.reset_index()

In [33]:
## 合并
ew = pd.merge(mnv_ew, ILLIQE_ew, on = 'index', how = 'left')
ew.columns = ['index', 'var_ew', 'skew_ew', 'kurt_ew', 'ILLIQ_ew']

## 市场平均方差、偏度、峰度：市值加权法

In [35]:
## 市值加权法
mnv_vw = pd.read_csv('Weight and Ave/mnv_vw.csv')
mnv_vw = mnv_vw.reset_index()
mnv_vw = mnv_vw[['index', 'var_w', 'mathskew_w', 'mathkurt_w']]
mnv_vw.columns = ['index', 'ave_var', 'ave_skew', 'ave_kurt']

In [38]:
## 等权重加权法下的市场非流动性
ILLIQE_vw = pd.read_csv("Rgrss/ILLIQE_vw.csv")
ILLIQE_vw = ILLIQE_vw.reset_index()

In [40]:
## 合并
vw = pd.merge(mnv_vw, ILLIQE_vw, on = 'index', how = 'left')
vw.columns = ['index', 'var_vw', 'skew_vw', 'kurt_vw', 'ILLIQ_vw']

## 合并等权重数据与市值加权数据

In [14]:
## 读取r0数据
r0 = pd.read_csv("Rgrss/r0.csv")
r0 = r0.reset_index()
r0.columns = ['index', 'r0']

In [8]:
## 读取牛市熊市虚拟变量
mkt_cc = pd.read_csv("Rgrss/mkt_cycle.csv")
mkt_cc = mkt_cc[['mktcc']]
mkt_cc = mkt_cc.reset_index()

In [46]:
reg_data = pd.merge(mkt_exr, mkt_prdct, on = 'index', how = 'left')
reg_data = pd.merge(reg_data, ew, on = 'index', how = 'left')
reg_data = pd.merge(reg_data, vw, on = 'index', how = 'left')
reg_data = pd.merge(reg_data, r0.head(144), on = 'index', how = 'left')
reg_data = pd.merge(reg_data, fmacro, on = 'index', how = 'left')
reg_data = pd.merge(reg_data, mkt_cc, on = 'index', how = 'left')
reg_data = reg_data.drop(['index'],axis=1)

In [13]:
reg_data.to_csv("Plot/reg_data.csv", index = False) 

In [14]:
reg_data.head(96).to_csv("Plot/reg_train.csv", index = False) 

In [12]:
reg_data

,r,year,mkt_var,mkt_skew,mkt_kurt,var_ew,skew_ew,kurt_ew,ILLIQ_ew,var_vw,skew_vw,kurt_vw,ILLIQ_vw,r0,DM,LR,FE,GDPGR,mktcc
0,0.072548,2007,0.019452,-0.838896,0.203171,0.044231,-0.143571,-0.136528,-6.252330,0.038083,-0.230598,-0.060531,-7.841304,0.160759,1.735738,0.0558,0.342000,0.138,1
1,0.096227,2007,0.003652,-2.066747,3.529964,0.014582,-0.535110,0.761365,-6.447494,0.014194,-0.499410,0.358401,-7.806725,0.072548,1.840684,0.0558,0.201600,0.138,1
2,0.257406,2007,0.003059,-0.664393,-0.448320,0.037484,0.122737,0.219057,-6.344277,0.017315,0.261622,0.065908,-7.952840,0.096227,1.847208,0.0567,0.223800,0.138,1
3,0.070527,2007,0.006224,-1.998115,4.018785,0.136230,-0.232924,0.676538,-6.133387,0.027041,-0.218136,0.401851,-7.990631,0.257406,1.876968,0.0567,0.365700,0.150,1
4,-0.118231,2007,0.011132,-1.810055,2.828659,0.056438,-0.358321,0.198198,-5.427716,0.026346,-0.270461,0.823381,-7.846217,0.070527,1.837965,0.0585,0.342100,0.150,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
139,-0.013986,2018,0.005955,1.064116,0.355968,0.016769,0.320492,0.441159,NaN,0.013004,0.354465,0.348423,NaN,-0.090724,2.322661,0.0435,0.039838,0.067,0
140,-0.118224,2018,0.001182,0.240583,-0.934328,0.008666,0.042578,0.391388,NaN,0.006360,0.344977,0.774344,NaN,-0.013986,2.345251,0.0435,0.019597,0.067,0
141,-0.005642,2018,0.008161,-0.274263,0.232362,0.024966,-0.455963,0.712175,NaN,0.017640,-0.326911,0.220151,NaN,-0.118224,2.324324,0.0435,-0.031237,0.065,0
142,-0.078632,2018,0.004063,-0.373812,0.710656,0.019460,-0.061371,0.706875,NaN,0.009962,0.003506,0.632958,NaN,-0.005642,2.336117,0.0435,-0.053625,0.065,0
